# 04. 탐색적 분석 — 전체 11단계 중 4단계

작업하며 주고받은 글(내가 셀마다 적은 칸과 클로드의 답변)은 빼고 코드 · 결과 · 결정만 남긴 판이다. 진행 방식은 README 「진행 방식」에 있다.

**이 단계에서 한 일과 정한 것** — 무엇이 LNG 발전량을 늘리거나 줄이는지 그림과 상관분석(두 값이 함께 늘고 주는 정도를 −1 ~ 1의 상관계수로 나타내는 것)으로 봤다. LNG 발전량과 상관계수가 가장 큰 것은 몇 시인가와 주말인가였다. 기온은 18 ~ 20도쯤에서 LNG 발전량이 가장 적고, 그보다 춥거나 더울수록 늘었다. 낮에는 태양광 발전량이 많을수록 LNG 발전량이 줄었다. 석탄 발전량은 LNG 발전량보다 먼저 바뀌지 않고 같은 시각에 함께 정해져서, 입력에 넣지 않기로 한 것을 그대로 두었다.

# 준비 ①

3단계가 만든 정제된 표를 읽어 옵니다. 아래 물음 둘이 다 이 표를 씁니다. 결과를 거의 안 찍으므로 적는 칸을 두지 않았습니다.

In [254]:
# 4단계 탐색적 분석 · 준비 ①
import os                                     # 폴더·파일 경로를 다루는 기본 모듈
import numpy as np                            # 숫자 계산
import pandas as pd                           # 표(데이터프레임)를 다루는 라이브러리
import matplotlib.pyplot as plt               # 그림 그리는 라이브러리
plt.rcParams["font.family"] = "AppleGothic"   # 맥에서 그림 속 한글이 깨지지 않게
plt.rcParams["axes.unicode_minus"] = False    # 마이너스 부호가 깨지지 않게
pd.set_option("display.width", 200)           # 표가 줄바꿈 없이 넓게 보이게
pd.set_option("display.max_columns", 50)      # 열이 많아도 생략하지 않게
pd.set_option("display.unicode.east_asian_width", True)   # 글자로 찍을 때만 쓰인다 — 표는 아래 display로 그린다
from IPython.display import display        # 표를 글자가 아니라 칸이 그려진 격자로 그려 준다

ROOT = os.getcwd()                            # 지금 폴더(보통 notebooks/)에서 시작해
while not os.path.isdir(os.path.join(ROOT, "data")) and ROOT != os.path.dirname(ROOT):
    ROOT = os.path.dirname(ROOT)              # data 폴더가 보일 때까지 위로 = 프로젝트 루트
RAW = os.path.join(ROOT, "data", "raw")       # 원자료 폴더 — 물음 3에서 석탄을 여기서 읽는다
HOT = os.path.join(ROOT, "data", "hot")       # 3단계가 저장한 정제된 표가 있는 폴더

tbl = pd.read_csv(os.path.join(HOT, "clean_2023_2024_2025.csv"), encoding="utf-8-sig")   # 3단계의 결과물
tbl["ts"] = pd.to_datetime(tbl["ts"])         # 글자로 저장된 시각을 다시 시각 형식으로

print("표 모양 (행, 열):", tbl.shape)
print("열 이름:", ", ".join(tbl.columns))
print()
print("해별 행 수 — 2024년은 2월 29일과 8월 한 달을 3단계에서 뺐다")
print(tbl.groupby("year").size().to_string())

표 모양 (행, 열): (25536, 13)
열 이름: ts, lng_mwh, nuke_mwh, solar_cap_mw, ta, hm, icsr, year, month, hour24, dow, is_weekend, solar_mwh

해별 행 수 — 2024년은 2월 29일과 8월 한 달을 3단계에서 뺐다
year
2023    8760
2024    8016
2025    8760


# 물음 1 — 지금 표의 13열이면 충분한가

> **결정** — 더 올릴 열은 없다. 풍력·바이오매스·연료전지는 후보에서 빼고, 2026년 태양광은 일사로 예측해서 넣는 방향(ⓒ)으로 간다

## 1-가

**무엇을 보는 셀인가.** 열마다 이 표에서 무슨 값인지, 빈 칸이 있는지, 값이 어디부터 어디까지인지를 한 표에 놓습니다. 「맞힐 값」은 LNG 하나이고, 「입력 후보」는 원자료에서 받아 온 것, 「파생」은 시각 하나에서 달력으로 만들어 낸 것입니다.

In [255]:
# 4단계 탐색적 분석 · 1-가
설명 = {"ts": "시각 · 1시간마다",          "lng_mwh": "맞힐 값 · LNG",          # 열마다 이 표에서 무슨 값인지
        "nuke_mwh": "입력 후보 · 원자력",   "solar_cap_mw": "입력 후보 · 태양광 설비",
        "solar_mwh": "입력 후보 · 태양광 발전", "ta": "입력 후보 · 기온",
        "hm": "입력 후보 · 습도",          "icsr": "입력 후보 · 일사",
        "year": "파생 · 해",               "month": "파생 · 월",
        "hour24": "파생 · 시각",           "dow": "파생 · 요일",
        "is_weekend": "파생 · 주말"}

요약 = pd.DataFrame({"무엇인가": pd.Series(설명)}).loc[tbl.columns]   # 표에 있는 열 순서대로 늘어놓는다
요약["빈칸"] = tbl.isna().sum()                      # 열마다 빈 칸이 몇 개인가
숫자 = tbl.select_dtypes("number")                   # 시각 열을 빼고 숫자 열만
요약["가장 작게"] = 숫자.min().round(1)               # 그 열이 가진 가장 작은 값
요약["가장 크게"] = 숫자.max().round(1)               # 가장 큰 값

print("지금 표에 있는 13열 — 무엇이고, 빈 칸이 있는지, 값이 어디까지인지")
display(요약.fillna(""))

지금 표에 있는 13열 — 무엇이고, 빈 칸이 있는지, 값이 어디까지인지


,무엇인가,빈칸,가장 작게,가장 크게
ts,시각 · 1시간마다,0,,
lng_mwh,맞힐 값 · LNG,0,4837.8,38033.3
nuke_mwh,입력 후보 · 원자력,0,15710.4,24415.7
solar_cap_mw,입력 후보 · 태양광 설비,0,6916.0,10834.0
ta,입력 후보 · 기온,0,-17.2,37.6
hm,입력 후보 · 습도,0,13.0,100.0
icsr,입력 후보 · 일사,0,0.0,3.8
year,파생 · 해,0,2023.0,2025.0
month,파생 · 월,0,1.0,12.0
hour24,파생 · 시각,0,0.0,23.0


## 1-나

**무엇을 보는 셀인가.** 기상청 ASOS 파일에는 열이 38개 들어 있고 지금 쓰는 것은 기온(ta)·습도(hm)·일사(icsr) 셋뿐입니다. 나머지 중 값이 들어 있는 열을 빈 칸 비율이 낮은 순서로 놓습니다. 빈 칸이 많은 것은 대개 그런 날씨가 아니어서 비는 것입니다 — 적설은 눈이 안 오면 비고, 일조시간은 밤에 빕니다. 2023년 1월부터 2026년 8월까지 44개월 파일을 다 읽어 셉니다.

In [ ]:
# 4단계 탐색적 분석 · 1-나
뜻 = {"ws": "풍속 m/s",   "wd": "풍향",        "pv": "증기압",    "td": "이슬점온도",   # 기상청이 이 파일에 담아 주는 열들
      "pa": "현지기압",   "ps": "해면기압",     "ss": "일조시간 h", "dc10Tca": "전운량 0~10",
      "dc10LmcsCa": "중하층운량", "vs": "시정 10m", "ts": "지면온도",
      "m005Te": "5cm 지중온도",  "m01Te": "10cm 지중온도",
      "m02Te": "20cm 지중온도",  "m03Te": "30cm 지중온도",
      "rn": "강수량 mm",  "dsnw": "적설 cm",    "lcsCh": "최저운고", "hr3Fhsc": "3시간 신적설"}

d = os.path.join(RAW, "asos")                                     # 기상 원본 폴더
asos = pd.concat([pd.read_csv(os.path.join(d, f), encoding="utf-8-sig")   # 44개월 파일을
                  for f in sorted(os.listdir(d)) if f.endswith(".csv")],  # 전부 읽어
                 ignore_index=True)                                       # 위아래로 붙인다

표 = pd.DataFrame([{"열": c, "무엇인가": n,                        # 열 이름과 뜻을 나란히 놓고
                    "빈칸%": round(asos[c].isna().mean() * 100, 1)}  # 빈 칸이 몇 퍼센트인지 센다
                   for c, n in 뜻.items()]).set_index("열").sort_values("빈칸%")

print("기상 원본 38열 중 지금 안 가져온 것 —", len(asos), "행에서 셌다")
display(표)

기상 원본 38열 중 지금 안 가져온 것 — 32136 행에서 셌다


,무엇인가,빈칸%
열,,
vs,시정 10m,0.0
pv,증기압,0.0
td,이슬점온도,0.0
pa,현지기압,0.0
ps,해면기압,0.0
dc10Tca,전운량 0~10,0.0
dc10LmcsCa,중하층운량,0.0
ts,지면온도,0.0
m03Te,30cm 지중온도,0.2


## 1-다

**무엇을 보는 셀인가.** 2025년 연간 파일에는 연료가 26종 들어 있습니다. 그중 발전량이 큰 열둘을 한 해 합계와 함께 놓고, 지금 표에 들어온 것인지와 2024년에도 따로 받을 수 있는지를 같이 찍습니다. 「24년에 없다」는 것은 2023·2025년에는 그 연료가 따로 있는데 2024년에는 다른 연료에 묶여 있어 세 해를 같은 기준으로 맞출 수 없다는 뜻입니다. 나머지 14종은 한 줄로 합계만 찍습니다.

In [ ]:
# 4단계 탐색적 분석 · 1-다
def load_kpx(year):                                                      # 연간 파일 한 해를 읽어 열 이름을 통일한다
    folder = os.path.join(RAW, "kpx")
    name = [f for f in os.listdir(folder) if f"{year}1231" in f][0]       # 파일명에 20251231 같은 기준일이 들어 있다
    df = pd.read_csv(os.path.join(folder, name), encoding="utf-8-sig")
    df.columns = ["date", "hour", "fuel", "cap_mw", "mwh"]               # 해마다 열 이름이 달라서 순서로 통일
    df["fuel"] = df["fuel"].str.strip()                                   # 연료 이름 앞뒤 공백 제거
    return df

k25, k24 = load_kpx(2025), load_kpx(2024)
있는것24 = set(k24["fuel"].unique()) | {"LNG", "무연탄", "수력", "양수", "원자력", "유연탄", "유전"}
# 2024년은 연간 파일(재생에너지 계열)과 API(연료 여덟 종) 둘을 합쳐야 한 해가 채워진다

합계 = (k25.groupby("fuel")["mwh"].sum() / 1000).round(0).sort_values(ascending=False)   # MWh를 GWh로
표 = pd.DataFrame({"25년 합계 GWh": 합계})
표["지금 표에"] = ["쓴다" if f in ("LNG", "원자력", "태양광") else "" for f in 표.index]
표["24년에"] = ["있다" if f in 있는것24 else "없다" for f in 표.index]

print("2025년 연간 파일의 연료 26종 중 발전량이 큰 열둘")
display(표.head(12))
print(f"\n나머지 14종을 다 더하면 {합계.iloc[12:].sum():,.0f} GWh")

2025년 연간 파일의 연료 26종 중 발전량이 큰 열둘


,25년 합계 GWh,지금 표에,24년에
fuel,,,
원자력,175549.0,쓴다,있다
유연탄,163584.0,,있다
LNG,159227.0,쓴다,있다
태양광,12523.0,쓴다,있다
연료전지,8351.0,,있다
바이오매스,7692.0,,있다
양수,4414.0,,있다
풍력,3646.0,,있다
수력,3175.0,,있다



나머지 14종을 다 더하면 5,232 GWh


## 1-라

**무엇을 보는 셀인가.** 1-나에서 이름만 봤던 기상 열들을 표에 실제로 붙여서 LNG와의 상관계수를 냅니다. 지금 쓰고 있는 기온·습도·일사도 맨 위에 같이 넣어 견줄 수 있게 했습니다.

In [ ]:
# 4단계 탐색적 분석 · 1-라
새열 = ["ws", "pv", "td", "pa", "ps", "ss", "dc10Tca", "dc10LmcsCa", "vs", "rn", "dsnw"]
뜻 = {"ta": "기온(쓰는 중)", "hm": "습도(쓰는 중)", "icsr": "일사(쓰는 중)", "ws": "풍속",
      "pv": "증기압", "td": "이슬점", "pa": "현지기압", "ps": "해면기압", "ss": "일조시간",
      "dc10Tca": "전운량", "dc10LmcsCa": "중하층운량", "vs": "시정", "rn": "강수량", "dsnw": "적설"}

d = os.path.join(RAW, "asos")                                        # 기상 원본을 다시 읽어
asos = pd.concat([pd.read_csv(os.path.join(d, f), encoding="utf-8-sig")
                  for f in sorted(os.listdir(d)) if f.endswith(".csv")], ignore_index=True)
asos["ts"] = pd.to_datetime(asos["tm"])                              # 글자로 된 시각을 시각 형식으로
b = tbl.merge(asos[["ts"] + 새열], on="ts", how="left")               # 같은 시각끼리 옆에 붙인다
for c in ("rn", "dsnw", "ss"):
    b[c] = b[c].fillna(0)                                            # 비·눈·해가 없어서 빈 것이므로 0

학습 = b[b["year"] != 2025]                                           # 2025는 채점용이라 보지 않는다
추움 = 학습[학습["ta"] <= 0]                                           # 그중 기온 0도 이하인 시각만

열들 = ["ta", "hm", "icsr"] + 새열
표 = pd.DataFrame({"무엇인가": [뜻[c] for c in 열들]}, index=열들)
표["빈칸%"] = [round(학습[c].isna().mean() * 100, 1) for c in 열들]
표["상관"] = [round(학습[c].corr(학습["lng_mwh"]), 3) for c in 열들]        # 학습 기간 전체에서
표["0도이하"] = [round(추움[c].corr(추움["lng_mwh"]), 3) for c in 열들]     # 기온 0도 이하에서만

print(f"LNG와의 상관 — 학습 기간(2023+2024) {len(학습)}시각, 그중 기온 0도 이하 {len(추움)}시각")
display(표)

LNG와의 상관 — 학습 기간(2023+2024) 16776시각, 그중 기온 0도 이하 2030시각


,무엇인가,빈칸%,상관,0도이하
ta,기온(쓰는 중),0.0,-0.060,-0.160
hm,습도(쓰는 중),0.0,-0.142,-0.144
icsr,일사(쓰는 중),0.0,0.066,0.224
ws,풍속,0.4,0.135,0.103
pv,증기압,0.0,-0.014,-0.197
td,이슬점,0.0,-0.103,-0.204
pa,현지기압,0.0,0.060,0.052
ps,해면기압,0.0,0.062,0.056
ss,일조시간,0.0,0.107,0.287
dc10Tca,전운량,0.0,0.011,0.025


## 1-마

**무엇을 보는 셀인가.** 연료마다 2023·2024·2025·2026년에 따로 받을 수 있는지를 한 표에 놓습니다. 칸에 들어가는 말 셋의 뜻은 이렇습니다.

- **받는다** — 그 해 자료에 그 연료가 따로 들어 있다.
- **묶임** — 2026년 API가 그 연료를 「신재생·기타」라는 한 덩어리로 묶어 줘서 따로 떼어낼 수 없다.
- **없다** — 그 해 자료에 아예 없다.

2023·2025년은 연간 파일 하나에 다 들어 있고, 2024년은 연간 파일(재생에너지 계열 열세 종)과 API(여덟 종)를 합쳐야 한 해가 채워집니다. 2026년은 API뿐입니다. 중유는 API에서 「유전」이라는 다른 이름으로 오는데 같은 것이라 「받는다」로 셌습니다.

맨 오른쪽 「네 해 다」에 ○가 붙은 연료만 입력 후보가 될 수 있습니다. 네 해 중 한 해라도 따로 못 받으면, 그 해에는 모델에 넣을 값이 없기 때문입니다.

In [ ]:
# 4단계 탐색적 분석 · 1-마
def load_kpx_fuels(year):                                            # 그 해 연간 파일에 들어 있는 연료 이름만 뽑는다
    folder = os.path.join(RAW, "kpx")
    n = [f for f in os.listdir(folder) if f"{year}1231" in f][0]      # 파일명에 20231231 같은 기준일이 들어 있다
    d = pd.read_csv(os.path.join(folder, n), encoding="utf-8-sig")
    d.columns = ["date", "hour", "fuel", "cap_mw", "mwh"]            # 해마다 열 이름이 달라서 순서로 통일
    return set(d["fuel"].str.strip().unique())

파일 = {y: load_kpx_fuels(y) for y in (2023, 2024, 2025)}
API = set(pd.read_csv(os.path.join(RAW, "api", "20240115.csv"),      # API가 주는 연료 여덟 종
                      encoding="utf-8-sig")["fuelTpCd"].unique())

다른이름 = {"중유": "유전"}                                            # 같은 것을 API는 다른 이름으로 준다
묶인것 = {"태양광", "풍력", "연료전지", "바이오매스"}                    # API는 이 넷을 「신재생·기타」 하나로 묶는다

def 표시(f, 있는것):
    if f in 있는것 or 다른이름.get(f) in 있는것:                        # 그 이름으로든 다른 이름으로든 있으면
        return "받는다"
    return "묶임" if f in 묶인것 else "없다"

연료 = ["LNG", "원자력", "유연탄", "무연탄", "태양광", "풍력",
        "양수", "수력", "중유", "연료전지", "바이오매스"]
표 = pd.DataFrame(index=연료)
표.index.name = "연료"
표["23 파일"] = [표시(f, 파일[2023]) for f in 연료]
표["24 파일+API"] = [표시(f, 파일[2024] | API) for f in 연료]           # 2024는 둘을 합쳐야 한 해가 된다
표["25 파일"] = [표시(f, 파일[2025]) for f in 연료]
표["26 API"] = [표시(f, API) for f in 연료]
표["네 해 다"] = ["○" if all(x == "받는다" for x in r) else "" for r in 표.values]

print("연료마다 해마다 따로 받을 수 있나")
print("  받는다 = 그 연료만 따로 있다 · 묶임 = 「신재생·기타」에 들어가 따로 못 뗀다 · 없다 = 그 해 자료에 없다")
display(표)

연료마다 해마다 따로 받을 수 있나
  받는다 = 그 연료만 따로 있다 · 묶임 = 「신재생·기타」에 들어가 따로 못 뗀다 · 없다 = 그 해 자료에 없다


,23 파일,24 파일+API,25 파일,26 API,네 해 다
연료,,,,,
LNG,받는다,받는다,받는다,받는다,○
원자력,받는다,받는다,받는다,받는다,○
유연탄,받는다,받는다,받는다,받는다,○
무연탄,받는다,받는다,받는다,받는다,○
태양광,받는다,받는다,받는다,묶임,
풍력,받는다,받는다,받는다,묶임,
양수,받는다,받는다,받는다,받는다,○
수력,받는다,받는다,받는다,받는다,○
중유,받는다,받는다,받는다,받는다,○


## 1-바

연료를 따로 예측해서 쓰려면 **그 연료를 무엇으로 맞힐 수 있는지**부터 봐야 합니다. 맞힐 재료가 없는 연료는 모델을 만들 수 없습니다. 그래서 만든 셀입니다.

**무엇을 보는 셀인가.** 표 둘입니다.

- **[표 1]** — 2026년에 「신재생·기타」로 묶여 오는 연료 넷(태양광·풍력·연료전지·바이오매스)이 일사·풍속·기온과 각각 얼마나 같이 움직이는지. 2023~2025년 전체에서 냅니다.
- **[표 2]** — 같은 넷의 2025년 한 해 합계와, 시각마다 값이 얼마나 흔들리는지(표준편차), 그리고 해가 없는 시각(0~3시·22~23시)에도 도는지.

표 1은 「이 연료를 기상으로 맞힐 수 있나」를 보는 것이고, 표 2는 「묶어 버리면 무엇이 흐려지나」를 보는 것입니다. 둘이 다른 물음이라 나눠 찍습니다.

In [ ]:
# 4단계 탐색적 분석 · 1-바
def 연료별_시간표(year):                                                  # 연간 파일에서 연료 넷의 시간별 발전량을 뽑아 한 표로
    folder = os.path.join(RAW, "kpx")
    n = [f for f in os.listdir(folder) if f"{year}1231" in f][0]
    d = pd.read_csv(os.path.join(folder, n), encoding="utf-8-sig")
    d.columns = ["date", "hour", "fuel", "cap_mw", "mwh"]
    d["fuel"] = d["fuel"].str.strip()
    d["ts"] = pd.to_datetime(d["date"]) + pd.to_timedelta(d["hour"].astype(int) - 1, unit="h")
    t = None
    for 연료, 이름 in [("태양광", "태양광"), ("풍력", "풍력"),
                       ("연료전지", "연료전지"), ("바이오매스", "바이오매스")]:
        한개 = d[d["fuel"] == 연료][["ts", "mwh"]].rename(columns={"mwh": 이름})
        t = 한개 if t is None else t.merge(한개, on="ts", how="left")
    return t

연료표 = pd.concat([연료별_시간표(y) for y in (2023, 2024, 2025)], ignore_index=True)

d = os.path.join(RAW, "asos")                                             # 기상을 같은 시각에 붙인다
asos = pd.concat([pd.read_csv(os.path.join(d, f), encoding="utf-8-sig")
                  for f in sorted(os.listdir(d)) if f.endswith(".csv")], ignore_index=True)
asos["ts"] = pd.to_datetime(asos["tm"])
연료표 = 연료표.merge(asos[["ts", "icsr", "ws", "ta"]], on="ts", how="left")
연료표["icsr"] = 연료표["icsr"].fillna(0)                                   # 일사는 밤에 비는 것이 정상
연료표["hour24"] = 연료표["ts"].dt.hour
연료표["year"] = 연료표["ts"].dt.year

연료들 = ["태양광", "풍력", "연료전지", "바이오매스"]
표1 = pd.DataFrame(index=연료들)
표1.index.name = "연료"
for 기상, 이름 in [("icsr", "일사와"), ("ws", "풍속과"), ("ta", "기온과")]:
    표1[이름] = [round(연료표[f].corr(연료표[기상]), 3) for f in 연료들]

밤 = 연료표[연료표["hour24"].isin([0, 1, 2, 3, 22, 23])]                    # 해가 없는 시각
올해 = 연료표[연료표["year"] == 2025]
표2 = pd.DataFrame(index=연료들)
표2.index.name = "연료"
표2["25년 합계 GWh"] = [round(올해[f].sum() / 1000) for f in 연료들]
표2["시간별 표준편차"] = [round(올해[f].std()) for f in 연료들]
표2["밤 시간 평균 MWh"] = [round(밤[밤["year"] == 2025][f].mean()) for f in 연료들]

print("[표 1] 연료마다 기상 셋과 얼마나 같이 움직이나 — 2023~2025년", len(연료표), "시각")
display(표1)
print("[표 2] 2025년 — 한 해 합계, 시각마다 값이 얼마나 흔들리나, 밤에도 도는가")
display(표2)

[표 1] 연료마다 기상 셋과 얼마나 같이 움직이나 — 2023~2025년 26304 시각


,일사와,풍속과,기온과
연료,,,
태양광,0.880,0.227,0.249
풍력,-0.175,0.358,-0.480
연료전지,-0.066,0.001,-0.124
바이오매스,-0.036,0.006,-0.071


[표 2] 2025년 — 한 해 합계, 시각마다 값이 얼마나 흔들리나, 밤에도 도는가


,25년 합계 GWh,시간별 표준편차,밤 시간 평균 MWh
연료,,,
태양광,12523,1826,11
풍력,3646,312,454
연료전지,8351,39,957
바이오매스,7692,112,880


# 물음 2 — 태양광 발전량과 LNG가 시간대별로 어떻게 맞물리나

> **결정** — 낮에는 태양광이 많을수록 LNG가 줄어든다. 그래서 태양광은 설비용량이 아니라 발전량으로 본다

## 2-가

**무엇을 보는 셀인가.** 평일만 남기고, 0시부터 23시까지 시각마다 태양광 발전량 평균과 LNG 발전량 평균을 해별로 놓습니다. 왼쪽 세 열이 태양광, 오른쪽 세 열이 LNG입니다. 주말을 뺀 것은 주말이 조업이 줄어 모양이 다르기 때문이고, 09-13에도 평일로 봤습니다.

In [ ]:
# 4단계 탐색적 분석 · 2-가
평일 = tbl[tbl["is_weekend"] == 0]                          # 토·일을 빼고 평일만 남긴다

태양 = 평일.pivot_table(index="hour24", columns="year",     # 행은 0~23시, 열은 해
                        values="solar_mwh", aggfunc="mean")  # 그 칸에 태양광 발전량 평균
엘엔지 = 평일.pivot_table(index="hour24", columns="year",
                          values="lng_mwh", aggfunc="mean")  # 같은 방식으로 LNG 발전량 평균

표 = pd.concat([태양.round(0).astype(int),                   # 두 표를 좌우로 붙인다
                엘엔지.round(0).astype(int)], axis=1)
표.columns = ["태양23", "태양24", "태양25", "LNG23", "LNG24", "LNG25"]   # 열 이름을 짧게
표.index.name = "시"

print("평일 시각별 평균 (MWh) — 왼쪽 셋이 태양광 발전량, 오른쪽 셋이 LNG 발전량")
display(표)

평일 시각별 평균 (MWh) — 왼쪽 셋이 태양광 발전량, 오른쪽 셋이 LNG 발전량


,태양23,태양24,태양25,LNG23,LNG24,LNG25
시,,,,,,
0,9,2,2,14158,15000,15118
1,2,1,1,12668,13406,13264
2,1,1,2,12170,13014,12790
3,1,1,2,12041,12963,12739
4,1,0,1,12293,13248,12969
5,5,7,6,13287,14262,13932
6,95,115,136,15544,16417,15993
7,466,519,638,18519,19435,19185
8,1250,1340,1643,21251,22430,21925


## 2-나

**무엇을 보는 셀인가.** 평일 낮 10~15시만 남기고, 거기서 다시 기온이 15~25도인 시각만 남긴 뒤, 태양광 발전량을 1,000 MWh 단위로 끊어 구간마다 LNG 평균을 봅니다.

In [ ]:
# 4단계 탐색적 분석 · 2-나
낮 = tbl[(tbl["is_weekend"] == 0)                    # 평일이고
         & (tbl["hour24"].between(10, 15))           # 10시부터 15시 사이이고
         & (tbl["ta"].between(15, 25))]              # 기온이 15~25도인 시각만 남긴다

구간 = pd.cut(낮["solar_mwh"],                        # 태양광 발전량을 구간으로 끊는다
              bins=[0, 1000, 2000, 3000, 4000, 5000, 8000])   # 1,000 MWh 단위, 마지막만 넓게

g = 낮.groupby(구간, observed=True).agg(              # 구간마다 아래 넷을 낸다
    시각수=("lng_mwh", "size"),                        # 그 구간에 시각이 몇 개 들어왔나
    LNG평균=("lng_mwh", "mean"),                       # 그 구간의 LNG 발전량 평균
    기온평균=("ta", "mean"),                            # 구간끼리 기온이 비슷한지 보려고
    원전평균=("nuke_mwh", "mean"))                      # 구간끼리 원전이 비슷한지 보려고

print("평일 낮 10~15시 · 기온 15~25도 — 태양광 발전량 구간별 LNG 평균 (MWh)")
display(g.round(0).astype({"LNG평균": int, "원전평균": int}))

평일 낮 10~15시 · 기온 15~25도 — 태양광 발전량 구간별 LNG 평균 (MWh)


,시각수,LNG평균,기온평균,원전평균
solar_mwh,,,,
"(0, 1000]",112,24061,21.0,19955
"(1000, 2000]",227,21658,21.0,19787
"(2000, 3000]",236,19148,20.0,19492
"(3000, 4000]",295,17198,21.0,19414
"(4000, 5000]",327,15663,20.0,19605
"(5000, 8000]",251,14359,20.0,20490


## 준비 ② — 원자료에서 두 열을 더 붙인다 (2-다와 물음 3에서 씁니다)

**무엇을 하는 셀인가.** 원자료를 다시 읽어 시각마다 두 값을 만들어 붙입니다. ⓐ `tot_mwh` — 그 시각에 거래된 전기 전체(모든 연료를 더한 것) ⓑ `coal_mwh` — 유연탄과 무연탄을 더한 석탄.

2023년과 2025년은 KPX 연간 파일에서 받고, **2024년만 API 일별 파일에서 받습니다.** 2024년 연간 파일에는 LNG·석탄·원자력이 없고 재생에너지 계열만 들어 있기 때문입니다. 3단계에서 뺀 시각(2024년 8월 한 달 · 2024-02-29)은 `tbl`에 아예 없으므로 그대로 따라 빠집니다. 그래서 2024년 평균은 여름 한 달이 빠진 값이고, 다른 두 해와 바로 비교하면 조금 낮게 나옵니다.

In [ ]:
# 4단계 탐색적 분석 · 준비 ② — 원자료에서 두 열을 더 붙인다
def load_kpx(year):                                           # KPX 연간 파일 한 해를 읽어 열 이름을 통일한다
    folder = os.path.join(RAW, "kpx")
    name = [f for f in os.listdir(folder) if f"{year}1231" in f][0]    # 파일명에 20231231 같은 기준일이 들어 있다
    df = pd.read_csv(os.path.join(folder, name), encoding="utf-8-sig")
    df.columns = ["date", "hour", "fuel", "cap_mw", "mwh"]    # 해마다 열 이름이 달라서 순서로 통일한다
    df["fuel"] = df["fuel"].str.strip()                       # 연료 이름 앞뒤 공백 제거
    df["ts"] = pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"].astype(int) - 1, unit="h")
    return df                                                 # 1~24시를 0~23시 시각으로 바꿔서 돌려준다

def load_api(year):                                           # API 일별 파일을 한 해치 모아 붙인다
    folder = os.path.join(RAW, "api")
    parts = []
    for f in sorted(os.listdir(folder)):
        if not (f.startswith(str(year)) and f.endswith(".csv")):
            continue                                          # 그 해 파일만 고른다
        q = os.path.join(folder, f)
        if os.path.getsize(q) < 100:
            continue                                          # 0행 파일 (2024-02-29 — API에 그날이 없다)
        one = pd.read_csv(q, encoding="utf-8-sig")
        if len(one) != 192:                                   # 정상은 연료 8종 × 24시간 = 192행
            continue                                          # 2024년 8월이 여기 걸려서 빠진다
        parts.append(one)
    a = pd.concat(parts, ignore_index=True)
    a = a.rename(columns={"fuelTpCd": "fuel", "amgo": "mwh"})  # 연간 파일과 열 이름을 같게 맞춘다
    a["ts"] = pd.to_datetime(a["tradeYmd"].astype(str)) + pd.to_timedelta(a["tradeNo"].astype(int) - 1, unit="h")
    return a

원자료 = pd.concat([load_kpx(2023), load_api(2024), load_kpx(2025)], ignore_index=True)   # 2024는 연간 파일에 LNG·석탄·원자력이 없다

총합 = 원자료.groupby("ts")["mwh"].sum()                        # 그 시각에 거래된 전기 전체
석탄 = 원자료[원자료["fuel"].isin(["유연탄", "무연탄"])].groupby("ts")["mwh"].sum()   # 석탄은 이 둘이다

tbl["tot_mwh"] = tbl["ts"].map(총합)                           # 시각을 열쇠로 tbl에 붙인다
tbl["coal_mwh"] = tbl["ts"].map(석탄)

print("tbl에 tot_mwh(총 전력거래량)와 coal_mwh(석탄)를 붙였습니다 — 행", len(tbl),
      "· 빈칸", int(tbl[["tot_mwh", "coal_mwh"]].isna().sum().sum()))
print("해별 시간평균 (MWh) — 2024년은 8월 한 달이 빠져 있어 조금 낮게 나옵니다")
display(tbl.groupby("year")[["tot_mwh", "lng_mwh", "coal_mwh"]].mean().round(0).astype(int))


tbl에 tot_mwh(총 전력거래량)와 coal_mwh(석탄)를 붙였습니다 — 행 25536 · 빈칸 0
해별 시간평균 (MWh) — 2024년은 8월 한 달이 빠져 있어 조금 낮게 나옵니다


,tot_mwh,lng_mwh,coal_mwh
year,,,
2023,62177,17525,20426
2024,61705,18288,17769
2025,62462,18177,18811


## 2-다

**무엇을 보는 셀인가.** 평일만 남기고 0시부터 23시까지 시각마다, 왼쪽 셋에 **그 시각에 거래된 전기 전체**를 해별로, 오른쪽 셋에 **그중 LNG가 차지한 몫(%)**을 해별로 놓습니다. 2-가와 같은 모양의 표이고 값만 바뀐 것입니다.

**무엇이 갈라지나.** 저녁에 LNG가 해마다 는 것을 놓고 세 가지가 가능합니다 — ⓐ 전체가 커져서 LNG도 같이 늘었다(몫은 그대로) ⓑ 전체는 그대로인데 LNG 몫이 커졌다 ⓒ 둘 다. 이 표에서 왼쪽과 오른쪽을 같이 보면 어느 쪽인지 보입니다.

In [ ]:
# 4단계 탐색적 분석 · 2-다
평일 = tbl[tbl["is_weekend"] == 0]                            # 토·일을 빼고 평일만 남긴다

총 = 평일.pivot_table(index="hour24", columns="year",          # 행은 0~23시, 열은 해
                      values="tot_mwh", aggfunc="mean")        # 그 칸에 총 전력거래량 평균
몫 = (평일.assign(몫=100 * 평일["lng_mwh"] / 평일["tot_mwh"])   # 그 시각 LNG가 전체에서 차지한 비율(%)
        .pivot_table(index="hour24", columns="year", values="몫", aggfunc="mean"))

표 = pd.concat([총.round(0).astype(int), 몫.round(1)], axis=1)  # 두 표를 좌우로 붙인다
표.columns = ["총23", "총24", "총25", "LNG몫23", "LNG몫24", "LNG몫25"]
표.index.name = "시"

print("평일 시각별 평균 — 왼쪽 셋은 그 시각에 거래된 전기 전체(MWh), 오른쪽 셋은 그중 LNG의 몫(%)")
display(표)


평일 시각별 평균 — 왼쪽 셋은 그 시각에 거래된 전기 전체(MWh), 오른쪽 셋은 그중 LNG의 몫(%)


,총23,총24,총25,LNG몫23,LNG몫24,LNG몫25
시,,,,,,
0,58267,57681,58536,24.3,26.0,25.9
1,56446,55693,56339,22.4,24.0,23.6
2,55893,55184,55732,21.7,23.5,23.0
3,55730,55132,55737,21.6,23.5,22.9
4,56046,55511,56054,21.9,23.8,23.2
5,57115,56599,57166,23.2,25.2,24.4
6,59593,59012,59546,26.0,27.8,26.8
7,63240,62698,63476,29.1,30.8,30.0
8,67730,67281,67761,31.1,33.0,32.0


## 2-라

2-다는 LNG 몫 하나만 보여 줍니다. 나머지 연료가 그 시각에 무엇을 하고 있었는지는 안 나옵니다.

**무엇을 보는 셀인가.** 2023년 평일만 남기고, 시각마다 **원자력·석탄·태양광·LNG·나머지가 각각 전체의 몇 %였는지**를 놓습니다. 맨 오른쪽에 그 시각의 전체 거래량을 같이 찍어서, 전체가 커질 때 어느 연료의 몫이 오르고 어느 연료의 몫이 내려가는지 같이 보이게 했습니다. 다섯 몫을 다 더하면 100%입니다.

**한 해만 쓰는 이유.** 하루 모양은 해마다 거의 같아서 한 해로 충분합니다. 2023년을 고른 것은 연간 파일 하나에 연료가 다 들어 있어 출처가 섞이지 않기 때문이고(2024년은 LNG·원자력은 API, 태양광은 연간 파일입니다), 2025년은 채점용이라 씁니다.

In [ ]:
# 4단계 탐색적 분석 · 2-라
k23 = load_kpx(2023)                                          # 2023년 연간 파일 — 연료가 한 파일에 다 있다
k23 = k23[k23["ts"].dt.dayofweek < 5]                         # 평일만 남긴다
k23["시"] = k23["ts"].dt.hour

def 몫(연료들):                                                 # 그 연료가 시각마다 전체의 몇 %였나
    갈래 = k23[k23["fuel"].isin(연료들)].groupby("시")["mwh"].sum()
    전체 = k23.groupby("시")["mwh"].sum()
    return (100 * 갈래 / 전체).round(1)

표 = pd.DataFrame({"원자력": 몫(["원자력"]),
                   "석탄": 몫(["유연탄", "무연탄"]),
                   "태양광": 몫(["태양광"]),
                   "LNG": 몫(["LNG"])})
표["나머지"] = (100 - 표.sum(axis=1)).round(1)                   # 수력·풍력·양수·부생가스 등
표["전체MWh"] = (k23.groupby("시")["mwh"].sum() / k23["ts"].dt.normalize().nunique()).round(0).astype(int)

print("2023년 평일 시각별 — 연료가 전체에서 차지한 몫(%)과 그 시각의 전체 거래량(MWh)")
display(표)


2023년 평일 시각별 — 연료가 전체에서 차지한 몫(%)과 그 시각의 전체 거래량(MWh)


,원자력,석탄,태양광,LNG,나머지,전체MWh
시,,,,,,
0,33.6,36.7,0.0,24.3,5.4,58267
1,34.7,37.7,0.0,22.4,5.2,56446
2,35.1,38.0,0.0,21.8,5.1,55893
3,35.2,38.0,0.0,21.6,5.2,55730
4,35.0,37.9,0.0,21.9,5.2,56046
5,34.3,37.3,0.0,23.3,5.1,57115
6,32.9,36.0,0.2,26.1,4.8,59593
7,31.0,34.2,0.7,29.3,4.8,63240
8,28.9,32.4,1.8,31.4,5.5,67730


## 2-마

지금까지 본 표는 2-가부터 2-라까지 전부 **하루 안의 모양**이었습니다. 「해마다 무엇이 달라졌나」는 나온 적이 없습니다.

**무엇을 보는 셀인가.** 한 해 전체 거래량(GWh)과 그 안에서 연료 여섯이 차지한 몫(%)을 2023년과 2025년으로 놓습니다. 맨 아래 줄이 2년 사이에 얼마나 움직였는지입니다. **8월은 세 해에서 똑같이 뺍니다** — 2-다에서 정한 규칙 그대로입니다.

**두 해만 쓰는 이유.** 2024년은 연료가 파일마다 갈려 있습니다 — 연간 파일에 LNG·석탄·원자력이 없고, API는 태양광을 「신재생·기타」로 묶어서 줍니다. 한 표에 섞으면 태양광 몫만 말이 안 되게 나옵니다.

In [ ]:
# 4단계 탐색적 분석 · 2-마
갈래 = {"원자력": ["원자력"], "석탄": ["유연탄", "무연탄"], "LNG": ["LNG"],
        "태양광": ["태양광"], "풍력": ["풍력"], "수력·양수": ["수력", "양수"]}

행 = []
for y in (2023, 2025):                                        # 연간 파일에 연료가 다 들어 있는 두 해
    d = load_kpx(y)
    d = d[d["ts"].dt.month != 8]                               # 2024년에 맞춰 세 해 모두에서 빼던 그 8월
    전체 = d["mwh"].sum()
    한줄 = {"해": y, "전체 GWh": round(전체 / 1000)}
    for 이름, 연료 in 갈래.items():
        한줄[이름] = round(100 * d[d["fuel"].isin(연료)]["mwh"].sum() / 전체, 1)
    행.append(한줄)

표 = pd.DataFrame(행).set_index("해")
표.loc["차이"] = (표.loc[2025] - 표.loc[2023]).round(1)          # 2년 사이에 얼마나 움직였나

print("8월을 뺀 한 해 전체 — 거래량(GWh)과 연료별 몫(%). 맨 아래는 2023 → 2025 차이")
display(표)
print("2024년은 연간 파일에 LNG·석탄·원자력이 없고 API는 태양광을 「신재생·기타」로 묶어 줘서 뺐습니다")


8월을 뺀 한 해 전체 — 거래량(GWh)과 연료별 몫(%). 맨 아래는 2023 → 2025 차이


,전체 GWh,원자력,석탄,LNG,태양광,풍력,수력·양수
해,,,,,,,
2023,493600.0,31.7,32.6,28.2,1.8,0.7,1.4
2025,494765.0,32.8,29.2,29.2,2.3,0.7,1.4
차이,1165.0,1.1,-3.4,1.0,0.5,0.0,0.0


2024년은 연간 파일에 LNG·석탄·원자력이 없고 API는 태양광을 「신재생·기타」로 묶어 줘서 뺐습니다


# 물음 3 — 석탄이 LNG보다 먼저 움직이나

> **결정** — 석탄은 LNG보다 먼저 움직이지 않는다(같은 시각에 같이 움직인다). 석탄을 입력에서 뺀 근거가 그대로 선다

## 3-가

**무엇을 보는 셀인가.** 석탄을 시간축에서 앞뒤로 밀어 가며 LNG와 얼마나 같이 움직이는지 봅니다. [표 1]은 −4시간부터 +4시간까지, [표 2]는 0개월부터 +3개월까지입니다. **시차가 +이면 「그만큼 앞선 석탄」과 「지금 LNG」를 맞춰 본 것**이고, 석탄이 먼저 움직인다면 + 쪽에서 상관이 가장 커집니다. 0에서 가장 크면 같은 시각에 같이 움직이는 것입니다.

**열이 둘인 이유.** 「값 자체 상관」은 석탄과 LNG의 발전량을 그대로 견준 것입니다. 이건 계절과 시간대가 둘 다를 같이 끌어올려서 높게 나오기 쉽습니다. 「변화량 상관」은 전 시각(전달) 대비 얼마나 움직였는지로 견준 것이라 그 공통 흐름이 빠집니다. 둘을 같이 놓고 봅니다.

**2025년을 뺀 이유.** 2025년은 채점용이라 입력을 고를 때 보면 안 됩니다. 09-20에 1-라를 학습 기간에서만 낸 것과 같습니다.

In [ ]:
# 4단계 탐색적 분석 · 3-가
학습 = tbl[tbl["year"].isin([2023, 2024])]                    # 2025는 채점용이라 여기서 보지 않는다

시각 = 학습.set_index("ts")[["lng_mwh", "coal_mwh"]].asfreq("h")   # 빠진 시각은 빈칸으로 둔다
변화 = 시각.diff()                                             # 전 시각 대비 얼마나 움직였나

표1 = pd.DataFrame({
    "시차(시간)": [f"{h:+d}" for h in range(-4, 5)],
    "변화량 상관": [round(변화["lng_mwh"].corr(변화["coal_mwh"].shift(h)), 3) for h in range(-4, 5)],
    "값 자체 상관": [round(시각["lng_mwh"].corr(시각["coal_mwh"].shift(h)), 3) for h in range(-4, 5)],
}).set_index("시차(시간)")

월 = 학습.groupby(["year", "month"], as_index=False)[["lng_mwh", "coal_mwh"]].mean()
월 = 월.sort_values(["year", "month"]).reset_index(drop=True)   # 달 순서대로 줄을 세운다
월변화 = 월[["lng_mwh", "coal_mwh"]].diff()                     # 전달 대비 얼마나 움직였나

표2 = pd.DataFrame({
    "시차(개월)": [f"{h:+d}" for h in range(0, 4)],
    "변화량 상관": [round(월변화["lng_mwh"].corr(월변화["coal_mwh"].shift(h)), 3) for h in range(0, 4)],
}).set_index("시차(개월)")

print("[표 1] 시간 단위 — 석탄을 h시간 뒤로 밀어 LNG와 맞춘 상관 (학습 2023+2024)")
display(표1)
print("[표 2] 달 단위 — 석탄을 h개월 뒤로 밀어 LNG와 맞춘 상관")
display(표2)
print("시차가 +이면 「h만큼 앞선 석탄」과 「지금 LNG」를 맞춘 것이다 — 석탄이 먼저면 +쪽에서 값이 가장 커진다")


[표 1] 시간 단위 — 석탄을 h시간 뒤로 밀어 LNG와 맞춘 상관 (학습 2023+2024)


,변화량 상관,값 자체 상관
시차(시간),,
-4,-0.132,0.368
-3,-0.037,0.376
-2,0.101,0.386
-1,0.263,0.392
+0,0.351,0.389
+1,0.282,0.373
+2,0.133,0.349
+3,-0.012,0.319
+4,-0.152,0.290


[표 2] 달 단위 — 석탄을 h개월 뒤로 밀어 LNG와 맞춘 상관


,변화량 상관
시차(개월),
+0,0.710
+1,0.333
+2,-0.318
+3,-0.402


시차가 +이면 「h만큼 앞선 석탄」과 「지금 LNG」를 맞춘 것이다 — 석탄이 먼저면 +쪽에서 값이 가장 커진다


## 3-나

**무엇을 보는 셀인가.** 평일 낮 10~15시, 기온 15~25도, 학습 기간(2023+2024)만 남기고, 석탄 발전량을 2,000 MWh 단위로 끊어 구간마다 LNG 평균을 봅니다. 기온·원전·태양광 평균을 같이 찍었습니다 — **구간마다 이 셋이 비슷해야 석탄만 다른 것이 됩니다.**

In [ ]:
# 4단계 탐색적 분석 · 3-나
낮 = tbl[(tbl["is_weekend"] == 0)                     # 평일이고
         & (tbl["hour24"].between(10, 15))            # 10시부터 15시 사이이고
         & (tbl["ta"].between(15, 25))                # 기온이 15~25도이고
         & (tbl["year"].isin([2023, 2024]))]          # 학습 기간인 시각만 남긴다

구간 = pd.cut(낮["coal_mwh"],                          # 석탄 발전량을 구간으로 끊는다
              bins=[0, 14000, 16000, 18000, 20000, 22000, 40000])

g = 낮.groupby(구간, observed=True).agg(
    시각수=("lng_mwh", "size"),                         # 그 구간에 시각이 몇 개 들어왔나
    LNG평균=("lng_mwh", "mean"),                        # 그 구간의 LNG 발전량 평균
    기온평균=("ta", "mean"),                             # 구간끼리 기온이 비슷한지
    원전평균=("nuke_mwh", "mean"),                       # 구간끼리 원전이 비슷한지
    태양광평균=("solar_mwh", "mean"))                     # 구간끼리 태양광이 비슷한지

print("평일 낮 10~15시 · 기온 15~25도 · 학습 2023+2024 — 석탄 발전량 구간별 (MWh)")
display(g.round(0).astype({"LNG평균": int, "원전평균": int, "태양광평균": int}))


평일 낮 10~15시 · 기온 15~25도 · 학습 2023+2024 — 석탄 발전량 구간별 (MWh)


,시각수,LNG평균,기온평균,원전평균,태양광평균
coal_mwh,,,,,
"(0, 14000]",228,15859,20.0,20507,4567
"(14000, 16000]",144,15876,20.0,19809,4029
"(16000, 18000]",183,16598,20.0,19631,3436
"(18000, 20000]",194,18319,20.0,19603,2512
"(20000, 22000]",101,20562,20.0,19509,2081
"(22000, 40000]",131,24713,23.0,19797,1498


## 3-다

**이 가설은 숫자로 확인됩니다.** 원자료에 발전량 말고 **설비용량**이 같이 들어 있습니다 — 3단계에서 태양광을 볼 때 썼던 그 열입니다. 설비용량은 그 연료로 최대 얼마나 만들 수 있는지이고, 발전량은 실제로 만든 것입니다. 둘을 나누면 여력이 남았는지 꽉 찼는지 나옵니다.

**무엇을 보는 셀인가.** 2023년 평일 시각마다 석탄·LNG·원자력이 **설비용량 중 몇 %를 실제로 돌렸는지**를 놓습니다. 100%에 가까우면 더 낼 데가 없는 것이고, 낮으면 여력이 남은 것입니다. 원자력은 비교하는 기준으로 같이 넣었습니다.

**읽는 법 한 줄.** 설비용량은 하루 내내 같은 값입니다(발전소를 낮에 더 짓지 않습니다). 그러니 이 표에서 움직이는 것은 발전량뿐입니다.

In [ ]:
# 4단계 탐색적 분석 · 3-다
k = load_kpx(2023)                                            # 설비용량 열이 있는 연간 파일
k = k[k["ts"].dt.dayofweek < 5]                                # 평일만
k["시"] = k["ts"].dt.hour

def 이용률(연료들):                                             # 설비용량 중 몇 %를 실제로 돌렸나
    d = k[k["fuel"].isin(연료들)].groupby(["ts", "시"], as_index=False).agg(
        설비=("cap_mw", "sum"), 발전=("mwh", "sum"))            # 시각마다 그 갈래를 합친 다음
    g = d.groupby("시").agg(설비=("설비", "mean"), 발전=("발전", "mean"))
    return (100 * g["발전"] / g["설비"]).round(1)               # 시각별 평균으로 이용률을 낸다

표 = pd.DataFrame({"석탄": 이용률(["유연탄", "무연탄"]),
                   "LNG": 이용률(["LNG"]),
                   "원자력": 이용률(["원자력"])})
설비 = {이름: round(k[k["fuel"].isin(연료)].groupby("ts")["cap_mw"].sum().mean())
        for 이름, 연료 in [("석탄", ["유연탄", "무연탄"]), ("LNG", ["LNG"]), ("원자력", ["원자력"])]}

print("2023년 평일 시각별 — 설비용량 중 실제로 돌린 비율(%)")
print("설비용량(MW): " + " · ".join(f"{k2} {v:,}" for k2, v in 설비.items()))
display(표)


2023년 평일 시각별 — 설비용량 중 실제로 돌린 비율(%)
설비용량(MW): 석탄 40,036 · LNG 43,559 · 원자력 24,704


,석탄,LNG,원자력
시,,,
0,53.4,32.5,79.3
1,53.1,29.1,79.3
2,53.0,27.9,79.3
3,52.9,27.6,79.3
4,53.0,28.2,79.3
5,53.2,30.5,79.3
6,53.5,35.7,79.3
7,54.0,42.5,79.3
8,54.8,48.8,79.3


## 3-라

**무엇을 보는 셀인가.** 달마다 석탄 이용률을 내서 2023년과 2025년을 나란히 놓습니다. 하루 안에서 5.8%p밖에 안 움직이던 값이 달 단위로는 얼마나 움직이는지 보는 것입니다.

**읽는 법 한 줄.** 3-다의 숫자(50.8~56.6%)와 비교하면 된다. 폭이 비슷하면 석탄은 어느 눈금으로 봐도 거의 고정인 것이고, 훨씬 넓으면 **하루가 아니라 달 단위로 조절되는 것**입니다.

**2024년을 뺀 이유.** 설비용량이 연간 파일에만 있고 2024년 연간 파일에는 석탄이 없습니다. 2-마와 같은 이유입니다.

In [ ]:
# 4단계 탐색적 분석 · 3-라
행 = []
for y in (2023, 2025):                                        # 설비용량이 들어 있는 연간 파일 두 해
    d = load_kpx(y)
    d = d[d["fuel"].isin(["유연탄", "무연탄"])]                  # 석탄만 남긴다
    d = d.assign(월=d["ts"].dt.month)
    시각별 = d.groupby(["ts", "월"], as_index=False).agg(
        설비=("cap_mw", "sum"), 발전=("mwh", "sum"))            # 시각마다 유연탄+무연탄을 합치고
    달별 = 시각별.groupby("월").agg(설비=("설비", "mean"),
                                  발전=("발전", "mean"))        # 달마다 평균을 낸 다음
    행.append((100 * 달별["발전"] / 달별["설비"]).round(1).rename(y))   # 이용률(%)로 바꾼다

표 = pd.concat(행, axis=1)
표["차이"] = (표[2025] - 표[2023]).round(1)                     # 2년 사이에 얼마나 움직였나

print("달마다 석탄이 설비용량 중 몇 %를 돌렸나 — 3-다의 시각별 폭(50.8 ~ 56.6%)과 견줘 보십시오")
display(표)


달마다 석탄이 설비용량 중 몇 %를 돌렸나 — 3-다의 시각별 폭(50.8 ~ 56.6%)과 견줘 보십시오


,2023,2025,차이
월,,,
1,58.8,40.9,-17.9
2,56.8,39.0,-17.8
3,41.8,29.5,-12.3
4,44.9,26.4,-18.5
5,45.0,32.8,-12.2
6,51.9,44.5,-7.4
7,58.5,63.1,4.6
8,59.7,65.1,5.4
9,49.1,60.6,11.5


## 3-마

**무엇을 보는 셀인가.** 3-라는 석탄만 놓았습니다. 같은 달에 LNG와 원자력이 무엇을 하고 있었는지, 그리고 그 달 전체 거래량이 달라졌는지가 안 나옵니다. 여기서 넷을 한 표에 놓습니다.

- 왼쪽 여섯 열 — 석탄·LNG·원자력이 달마다 설비용량 중 몇 %를 돌렸나. 2023년과 2025년을 나란히.
- 맨 오른쪽 한 열 — 그 달 전체 거래량이 2023년 대비 몇 % 달라졌나.

**읽는 법.** 석탄이 크게 내려간 달을 먼저 찾고, **같은 줄에서 오른쪽 세 값을 보십시오.**

- 전체 거래량이 줄었으면 — 쓸 일이 줄어서 석탄도 줄었을 수 있습니다.
- 전체는 그대로인데 LNG가 올라갔으면 — LNG가 채운 것입니다.
- 전체도 그대로이고 LNG도 그대로인데 원자력이 올라갔으면 — **원자력이 채운 것입니다.**

**원자력을 넣은 이유.** 1단계에서 원자력을 원인 후보로 적은 근거가 「LNG와 상관없이 자기 사정(정비 일정)으로 정해진다」였습니다. 그 말이 맞는지가 이 표에 나옵니다 — 원자력 이용률이 달마다 크게 움직이면 정비 일정이 그만큼 몰려 있다는 뜻입니다.

In [ ]:
# 4단계 탐색적 분석 · 3-마
def 월이용률(year, 연료들):                                     # 달마다 설비용량 중 몇 %를 돌렸나
    d = load_kpx(year)
    d = d[d["fuel"].isin(연료들)]
    d = d.assign(월=d["ts"].dt.month)
    시각별 = d.groupby(["ts", "월"], as_index=False).agg(
        설비=("cap_mw", "sum"), 발전=("mwh", "sum"))
    달별 = 시각별.groupby("월").agg(설비=("설비", "mean"), 발전=("발전", "mean"))
    return (100 * 달별["발전"] / 달별["설비"]).round(1)

def 월총량(year):                                               # 달마다 그 시각 전체 거래량 평균
    d = load_kpx(year)
    d = d.assign(월=d["ts"].dt.month)
    return d.groupby(["ts", "월"], as_index=False)["mwh"].sum().groupby("월")["mwh"].mean()

표 = pd.DataFrame({
    "석탄23": 월이용률(2023, ["유연탄", "무연탄"]), "석탄25": 월이용률(2025, ["유연탄", "무연탄"]),
    "LNG23": 월이용률(2023, ["LNG"]),             "LNG25": 월이용률(2025, ["LNG"]),
    "원전23": 월이용률(2023, ["원자력"]),          "원전25": 월이용률(2025, ["원자력"]),
})
표["총량변화%"] = (100 * (월총량(2025) / 월총량(2023) - 1)).round(1)   # 그 달 전체 거래량이 얼마나 달라졌나

print("달마다 설비용량 중 몇 %를 돌렸나 (2023 · 2025). 맨 오른쪽은 그 달 전체 거래량의 2년 변화")
display(표)


달마다 설비용량 중 몇 %를 돌렸나 (2023 · 2025). 맨 오른쪽은 그 달 전체 거래량의 2년 변화


,석탄23,석탄25,LNG23,LNG25,원전23,원전25,총량변화%
월,,,,,,,
1,58.8,40.9,48.4,44.5,81.3,87.8,-3.5
2,56.8,39.0,46.8,48.0,78.2,88.1,2.0
3,41.8,29.5,42.7,42.1,78.9,87.7,1.4
4,44.9,26.4,36.7,39.6,72.7,80.8,-0.3
5,45.0,32.8,35.0,33.9,75.0,78.0,-1.1
6,51.9,44.5,36.9,34.2,75.9,76.3,0.2
7,58.5,63.1,41.5,39.5,82.0,74.7,5.4
8,59.7,65.1,43.8,42.2,82.2,68.9,2.6
9,49.1,60.6,39.6,33.0,80.3,72.1,4.7
